# DeepSeek-OCR Inference on Google Colab

This notebook demonstrates how to use the DeepSeek-OCR model for optical character recognition and document understanding.

**Model:** [deepseek-ai/DeepSeek-OCR](https://huggingface.co/deepseek-ai/DeepSeek-OCR)

**Features:**
- Advanced OCR capabilities
- Document understanding
- Multi-scale feature extraction
- Adaptive resolution processing

**Note:** This model requires a GPU runtime. Go to Runtime → Change runtime type → T4 GPU

## 1. Installation

First, let's install the required dependencies.

In [ ]:
# Install required packages
!pip install -q torch transformers tokenizers accelerate Pillow sentencepiece protobuf addict

print("✓ Installation complete!")

## 2. Setup and Import Libraries

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from PIL import Image
import os

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: Running on CPU. This will be slow. Please enable GPU runtime.")

## 3. Load the DeepSeek-OCR Model

This will download the model from HuggingFace (approximately 10GB).

In [ ]:
model_name = "deepseek-ai/DeepSeek-OCR"

print(f"Loading tokenizer from {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

print(f"Loading model from {model_name}...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
).eval()

print("✓ Model loaded successfully!")

## 4. Upload Your Image

Upload an image file (JPG, PNG, etc.) containing text or documents you want to process.

In [ ]:
from google.colab import files
import shutil

# Upload image
print("Please upload an image file:")
uploaded = files.upload()

# Get the first uploaded file
image_path = list(uploaded.keys())[0]
print(f"\n✓ Uploaded: {image_path}")

# Display the image
from IPython.display import Image as IPImage, display
display(IPImage(image_path))

## 5. Run OCR Inference

Process the uploaded image using DeepSeek-OCR.

In [ ]:
# Create output directory
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

print(f"Running inference on {image_path}...")
print("This may take a minute...\n")

# Run inference
result = model.infer(
    tokenizer,
    image_file=image_path,
    output_path=output_dir
)

print("✓ Inference complete!")

## 6. Display Results

In [ ]:
print("=" * 80)
print("OCR RESULT:")
print("=" * 80)
print(result)
print("=" * 80)

## 7. Download Output Files (Optional)

Download any generated output files.

In [ ]:
import glob

# List all output files
output_files = glob.glob(os.path.join(output_dir, "*"))

if output_files:
    print("Available output files:")
    for f in output_files:
        print(f"  - {f}")
    
    # Download all files
    print("\nDownloading files...")
    for f in output_files:
        files.download(f)
    print("✓ Download complete!")
else:
    print("No output files generated.")

## 8. Batch Processing (Optional)

Process multiple images at once.

In [ ]:
# Upload multiple images
print("Upload multiple images for batch processing:")
uploaded_batch = files.upload()

image_paths = list(uploaded_batch.keys())
print(f"\n✓ Uploaded {len(image_paths)} images")

In [ ]:
# Process all images
batch_output_dir = "batch_output"
os.makedirs(batch_output_dir, exist_ok=True)

results = []

for i, img_path in enumerate(image_paths):
    print(f"\nProcessing {i+1}/{len(image_paths)}: {img_path}")
    
    img_output_dir = os.path.join(batch_output_dir, f"image_{i+1}")
    os.makedirs(img_output_dir, exist_ok=True)
    
    result = model.infer(
        tokenizer,
        image_file=img_path,
        output_path=img_output_dir
    )
    results.append(result)
    print(f"✓ Complete")

print(f"\n✓ All {len(results)} images processed!")

In [ ]:
# Display all results
for i, (img_path, result) in enumerate(zip(image_paths, results)):
    print(f"\n{'='*80}")
    print(f"RESULT {i+1}: {img_path}")
    print(f"{'='*80}")
    print(result)
    print(f"{'='*80}")

## Tips and Best Practices

1. **GPU Runtime**: Always use a GPU runtime (T4 or better) for optimal performance
2. **Image Quality**: Higher resolution images generally produce better OCR results
3. **Supported Formats**: The model works with common image formats (JPG, PNG, etc.)
4. **Memory Management**: If you encounter out-of-memory errors, try:
   - Processing images one at a time
   - Using smaller images
   - Restarting the runtime
5. **Output Files**: Check the output directory for additional generated files like markdown or visualizations

## Resources

- [DeepSeek-OCR on HuggingFace](https://huggingface.co/deepseek-ai/DeepSeek-OCR)
- [GitHub Repository](https://github.com/azman0101/DeepSeek-OCR-Inference)
- [Transformers Documentation](https://huggingface.co/docs/transformers/)
